# Figure 2 — linear transfer of $t_0$ (3 panels, fine-grained panel (c))

Downstream $t_0$ (epochs to memorize CIFAR-10 random labels) vs. the pre-training epoch it warm-started
from, across three memorization-to-memorization settings. Each panel is drawn under **equal aspect** so
the dashed slope $=-1$ guide sits at a true 45 degrees: a curve on the guide means "one pre-training
epoch buys exactly one downstream epoch". The red dashed line is the from-scratch baseline (the highest
$t_0$ at $k=0$). All $t_0$ fitted fresh from raw `metrics.json` with `common.fit_t0` (no cache).

- **(a)** upstream label-seed sweep (seeds 7/21/33/55) -> CIFAR-10 random labels.
- **(b)** split CIFAR-10 D1 -> disjoint D2 random labels, over 4 split seeds (37/11/29/53).
- **(c)** random pixels + random labels -> CIFAR-10 random labels, fine grid $k = 0 \dots 18$.

- **Consumes** (metrics under `figure_data/`, grouped by curve):
  `downstream_randlabel_from_labelseed{7,21,33,55}/init{0000..0040:5}`,
  `split_stage2_randlabel_splitseed{37,11,29,53}/init{0000..0035:5}`,
  `downstream_randlabel_from_randpixel_finegrained/init{0000..0018}`
- **Produces:** `outputs/fig2_linear_transfer.png`


In [ ]:
import sys
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FixedLocator

# repo root (the dir that holds common/) onto sys.path, from any working directory
_root = Path.cwd()
while not (_root / "common").is_dir() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))

from common.config import FIGURE_DATA_DIR, OUTPUT_DIR
from common.fit_t0 import fit_t0

# ---- panel run-dir templates (flat naming under figure_data/) ----
# panel (a): upstream label-seed sweep -> CIFAR-10 random labels
A_SEEDS = [7, 21, 33, 55]
A_EPOCHS = list(range(0, 41, 5))
a_path = lambda seed, k: FIGURE_DATA_DIR / f"downstream_randlabel_from_labelseed{seed}" / f"init{k:04d}" / "metrics.json"

# panel (b): split CIFAR-10, D1 -> disjoint D2 random labels, over 4 split seeds
B_SPLIT_SEEDS = [37, 11, 29, 53]
B_EPOCHS = list(range(0, 36, 5))
b_path = lambda ss, k: FIGURE_DATA_DIR / f"split_stage2_randlabel_splitseed{ss}" / f"init{k:04d}" / "metrics.json"

# panel (c): random pixels + random labels -> CIFAR-10 random labels, every epoch 0-18
C_EPOCHS = list(range(0, 19))
c_path = lambda k: FIGURE_DATA_DIR / f"downstream_randlabel_from_randpixel_finegrained" / f"init{k:04d}" / "metrics.json"


def t0_of(metrics_path):
    m = json.loads(Path(metrics_path).read_text(encoding="utf-8"))
    return fit_t0([r["epoch"] for r in m["epochs"]], [r["train_acc"] for r in m["epochs"]]).t0


# ---- OLS slope with standard error (for the fitted-slope legends) ----
def ols(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float); n = len(x)
    A = np.vstack([x, np.ones_like(x)]).T
    slope, intercept = np.linalg.lstsq(A, y, rcond=None)[0]
    resid = y - (slope * x + intercept)
    sxx = float(np.sum((x - x.mean()) ** 2))
    se = math.sqrt(float(np.sum(resid ** 2)) / (n - 2) / sxx) if n > 2 and sxx > 0 else float("nan")
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    r2 = 1 - float(np.sum(resid ** 2)) / ss_tot if ss_tot > 0 else float("nan")
    return dict(slope=float(slope), intercept=float(intercept), se=float(se), r2=float(r2), n=int(n))

In [ ]:
# fit every point fresh from raw metrics.json (no cache) into a tidy frame
records = []
for seed in A_SEEDS:
    for k in A_EPOCHS:
        records.append(dict(panel="a", curve_seed=seed, pretraining_epoch=k, t0=t0_of(a_path(seed, k))))
for ss in B_SPLIT_SEEDS:
    for k in B_EPOCHS:
        records.append(dict(panel="b", curve_seed=ss, pretraining_epoch=k, t0=t0_of(b_path(ss, k))))
for k in C_EPOCHS:
    records.append(dict(panel="c", curve_seed=42, pretraining_epoch=k, t0=t0_of(c_path(k))))
inv = pd.DataFrame(records)
print("fitted", len(inv), "points  |  (a)", (inv.panel == "a").sum(),
      " (b)", (inv.panel == "b").sum(), " (c)", (inv.panel == "c").sum())

# panel -> (k_min, k_max) of the segment used for the OLS slope
FIT_RANGES = {"a": (10, 40), "b": (10, 25), "c": (1, 3)}


def curve_of(panel, seed=None):
    m = (inv["panel"] == panel) if seed is None else ((inv["panel"] == panel) & (inv["curve_seed"] == seed))
    d = inv[m].sort_values("pretraining_epoch")
    return d["pretraining_epoch"].to_numpy(float), d["t0"].to_numpy(float)

# per-curve fitted slopes (what the legends quote)
for panel in ("a", "b"):
    lo, hi = FIT_RANGES[panel]
    for sd in sorted(inv.loc[inv.panel == panel, "curve_seed"].unique()):
        x, y = curve_of(panel, sd)
        mm = (x >= lo) & (x <= hi)
        f = ols(x[mm], y[mm])
        print(f"  ({panel}) seed {int(sd):>2d}: slope {f['slope']:+.3f}  R2 {f['r2']:.3f}  (fit {lo}-{hi})")

In [ ]:
FS_AXIS, FS_TICK, FS_LEG, FS_PANEL = 11, 9, 8.5, 11

plt.rcParams.update({
    'figure.dpi': 130, 'font.family': 'DejaVu Sans',
    'axes.labelsize': FS_AXIS, 'xtick.labelsize': FS_TICK, 'ytick.labelsize': FS_TICK,
    'legend.fontsize': FS_LEG,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.linewidth': 0.9,
    'axes.grid': False, 'pdf.fonttype': 42, 'ps.fonttype': 42,
})

A_SEEDS = sorted(inv.loc[inv.panel == 'a', 'curve_seed'].unique())
COLORS_A = plt.cm.viridis(np.linspace(0.15, 0.85, len(A_SEEDS)))
MARKERS = ['o', 's', '^', 'D']
# panel (c) reuses exactly panel (a)'s seed-21 colour (same upstream seed 21)
COLOR_SINGLE = COLORS_A[A_SEEDS.index(21)]

# panel (b): the four D1/D2 split seeds use a DISTINCT plasma ramp -- they are split seeds, not
# label seeds, so a different colormap stops the reader reading 'run i in (b)' = 'run i in (a)'.
B_SEEDS = sorted(inv.loc[inv.panel == 'b', 'curve_seed'].unique())
COLORS_B = plt.cm.plasma(np.linspace(0.1, 0.8, len(B_SEEDS)))

# panels drawn as several seed curves (both a and b on the viridis ramp). Panel (c) is a single curve.
MULTI_SEED = {'a': (A_SEEDS, COLORS_A), 'b': (B_SEEDS, COLORS_B)}

# Panels whose fitted window is straight enough to quote a slope. Panel (c) is excluded on purpose:
# on the fine grid it has no straight segment at all -- see the rolling-window table in Part 0c.
SLOPE_IN_LEGEND = {'b'}
# Legends name the fitted slope only -- no seed text (house style). Panel (a) shows its 4 per-seed
# slopes; panel (b) now shows its 4 per-split-seed slopes, so the reader can see at a glance whether
# every seed is steeper than -1. Panel (c) is left without a legend: one curve, no slope to quote.
LEGEND_PANELS = {'a', 'b'}

# Panel (c) now carries 19 points in a 1.58 in wide panel, i.e. ~5.1 pt per epoch. At the ms = 5.0 used
# by (a)/(b) the markers would touch and read as a bead chain, so (c) alone drops to ms = 3.2. Marker
# shape, line width, colour and style are unchanged. The measured check at the bottom of this cell
# prints marker diameter against marker spacing.
MS_MAIN, MS_FINE = 5.0, 3.2

# One line each. Measured at 11pt: (a) needs 1.84 in of a 2.83 in panel and (b) 2.62 of 3.06, so both
# fit outright. (c) needs 2.28 in of a 1.58 in panel, so its title overhangs the panel by ~0.35 in per
# side -- harmless, because GAP leaves more clearance than that. The check at the bottom of this cell
# fails loudly if a layout change ever makes two titles actually touch.
PANEL_TITLE = {'a': '(a) CIFAR-10 \u2192 CIFAR-10',
               'b': '(b) Split CIFAR-10 \u2192 Split CIFAR-10',
               'c': '(c) Random pixels \u2192 CIFAR-10'}
LEGEND_KW = dict(frameon=False, handlelength=1.9, borderpad=0.2, labelspacing=0.35, handletextpad=0.5)

# ---------------------------------------------------------------------------------------------
# 45 degrees, panel by panel.
#
# The guide reads as 45 degrees exactly when one epoch along x covers the same physical distance as
# one epoch along y. That is a *within-panel* condition: it says nothing about the three panels
# sharing a scale. So each panel gets
#   - the same drawing height H_AX, and a y range cropped to its own data;
#   - its own scale K_p = H_AX / y_range_p, applied to BOTH of its axes;
#   - width W_p = K_p * x_range_p, which follows.
#
# Equal heights + a shared baseline at t0 = 0 means the panel tops line up too, so the titles sit at
# one height without any special handling.
#
# What this trades away: K differs between panels, so a physical distance is not the same number of
# epochs across panels. That comparison was never the point -- the three panels are three different
# settings with x spans differing by 2x. What matters, "is this curve steeper or shallower than -1",
# is a within-panel reading, and it is exact in all three.
#
# Panel (c)'s fine grid does not change any of this: 19 points instead of 5 over the same k = 0..18
# span, and its own t0 max (41.4) still rounds to the same 0..45 y range, so the geometry is
# identical to the sparse version of this figure.
#
# constrained_layout / tight_layout are deliberately NOT used: they resize axes to fit labels, which
# is precisely what would destroy the 45 degrees.
# ---------------------------------------------------------------------------------------------
X_PAD_LO = 1.2                       # room so the k = 0 marker is not sliced by the spine
X_HI = {p: float(inv.loc[inv.panel == p, 'pretraining_epoch'].max()) + 3.0 for p in 'abc'}
XLIM = {p: (-X_PAD_LO, X_HI[p]) for p in 'abc'}
X_RANGE = {p: XLIM[p][1] - XLIM[p][0] for p in 'abc'}

# y cropped to each panel's own data, rounded up to a multiple of 5
Y_HI = {p: 5.0 * math.ceil((float(inv.loc[inv.panel == p, 't0'].max()) + 1.5) / 5.0) for p in 'abc'}
YLIM = {p: (0.0, Y_HI[p]) for p in 'abc'}

H_AX = 3.20                                                 # inches, identical for all three panels
K = {p: H_AX / Y_HI[p] for p in 'abc'}                      # inches per epoch, x and y alike
W_AX = {p: K[p] * X_RANGE[p] for p in 'abc'}
M_L, M_R, M_B, M_T, GAP = 0.80, 0.15, 0.62, 0.44, 0.55      # GAP leaves room for (c)'s wide title
FIG_W = M_L + sum(W_AX.values()) + 2 * GAP + M_R
FIG_H = M_B + H_AX + M_T

print('points    : ' + '   '.join(f'({p}) {int((inv.panel == p).sum())}' for p in 'abc'))
print('x limits  : ' + '   '.join(f'({p}) {XLIM[p][0]:.1f}..{XLIM[p][1]:.0f}' for p in 'abc'))
print('y limits  : ' + '   '.join(f'({p}) 0..{Y_HI[p]:.0f}' for p in 'abc'))
print('scale K   : ' + '   '.join(f'({p}) {K[p]:.4f} in/epoch' for p in 'abc'))
print('axes sizes: ' + '   '.join(f'({p}) {W_AX[p]:.2f} x {H_AX:.2f} in' for p in 'abc'))
print(f'figsize   : {FIG_W:.2f} x {FIG_H:.2f} in')

fig = plt.figure(figsize=(FIG_W, FIG_H))
axes = {}
x_cursor = M_L
for p in 'abc':
    axes[p] = fig.add_axes([x_cursor / FIG_W, M_B / FIG_H, W_AX[p] / FIG_W, H_AX / FIG_H])
    x_cursor += W_AX[p] + GAP

plot_rows = []

for panel in ('a', 'b', 'c'):
    ax = axes[panel]
    lo, hi = FIT_RANGES[panel]

    if panel in MULTI_SEED:
        seeds, colors = MULTI_SEED[panel]
        for i, sd in enumerate(seeds):
            x, y = curve_of(panel, sd)
            m = (x >= lo) & (x <= hi)
            f = ols(x[m], y[m])
            lab = f'{f["slope"]:.2f}'.replace('-', '\u2212')      # unicode minus
            ax.plot(x, y, marker=MARKERS[i % len(MARKERS)], ms=MS_MAIN, lw=1.5,
                    color=colors[i], mec='white', mew=0.5, zorder=3,
                    label=f'run {i + 1}')
            for xi, yi in zip(x, y):
                plot_rows.append(dict(panel=panel, curve=f'seed {int(sd)}', curve_seed=int(sd),
                                      pretraining_epoch=xi, t0=yi))
    else:
        # panel (c): a single unlabelled curve
        x, y = curve_of(panel)
        cseed = int(inv.loc[inv.panel == panel, 'curve_seed'].iloc[0])
        ax.plot(x, y, marker='o', ms=MS_FINE, lw=1.5, color=COLOR_SINGLE, mec='white',
                mew=0.4, zorder=3, label=None)
        for xi, yi in zip(x, y):
            plot_rows.append(dict(panel=panel, curve='single', curve_seed=cseed,
                                  pretraining_epoch=xi, t0=yi))

    # ---- from-scratch baseline + slope -1 guide, both anchored at the highest t0 at k = 0 here ----
    sub = inv[(inv.panel == panel) & (inv.pretraining_epoch == 0)]
    t0_max0 = float(sub['t0'].max())
    xref = np.linspace(0.0, XLIM[panel][1], 200)
    # both references mean the same thing in all three panels, so only (a) spends legend entries on them
    ax.plot([0.0, XLIM[panel][1]], [t0_max0, t0_max0], ls='--', color='#c44e52', lw=1.2, zorder=1,
            label='From-scratch baseline' if panel == 'a' else None)
    ax.plot(xref, t0_max0 - xref, ls='--', color='gray', lw=1.0, alpha=0.7, zorder=1, clip_on=True,
            label='slope = \u22121 (ref.)' if panel == 'a' else None)
    if panel in LEGEND_PANELS:
        ax.legend(loc='upper right', **LEGEND_KW)

    ax.set_xlim(*XLIM[panel])
    ax.set_ylim(*YLIM[panel])
    ax.xaxis.set_major_locator(FixedLocator(list(range(0, int(X_HI[panel]) + 1, 10))))
    ax.yaxis.set_major_locator(FixedLocator(list(range(0, int(Y_HI[panel]) + 1, 10))))
    ax.set_xlabel('Pre-training epoch')
    # every panel is labelled: the y ranges differ, so a reader cannot infer (b)/(c) from (a)
    ax.set_ylabel(r'$t_0$ (downstream epochs)' if panel == 'a' else '')
    ax.set_title(PANEL_TITLE[panel], fontsize=FS_PANEL, pad=6)

# ---- checklist item 1, measured rather than eyeballed ----
fig.canvas.draw()
print()
print('physical scale per panel (points per epoch). ratio must be 1.000 for the guide to sit at 45\u00b0:')
for p in 'abc':
    bb = axes[p].get_window_extent()
    sx, sy = bb.width / X_RANGE[p], bb.height / Y_HI[p]
    print(f'  panel ({p}):  x {sx:7.4f}   y {sy:7.4f}   ratio {sx / sy:.6f}   '
          f'guide angle {math.degrees(math.atan2(sy, sx)):5.2f}\u00b0')
    assert abs(sx / sy - 1.0) < 1e-6, f'panel ({p}) is not equal-aspect'

# ---- panel (c) marker spacing: markers must not touch ----
sx_c = axes['c'].get_window_extent().width / X_RANGE['c'] * 72 / fig.dpi   # pt per epoch
print()
print(f'panel (c): {sx_c:.2f} pt per epoch, marker diameter {MS_FINE:.1f} pt '
      f'-> gap {sx_c - MS_FINE:.2f} pt between neighbouring markers')
assert sx_c > MS_FINE, 'panel (c) markers overlap; lower MS_FINE'

# ---- titles must not collide; (c)'s legitimately overhangs its narrow panel ----
rend = fig.canvas.get_renderer()
tex = {p: axes[p].title.get_window_extent(renderer=rend) for p in 'abc'}
ax_ = {p: axes[p].get_window_extent() for p in 'abc'}
print()
print('title widths vs panel widths (points), and clearance between neighbouring titles:')
for p in 'abc':
    over = (tex[p].width - ax_[p].width) / 2
    print(f'  panel ({p}): title {tex[p].width:6.1f}  panel {ax_[p].width:6.1f}  '
          f'overhang per side {over:+6.1f}')
for lft, rgt in (('a', 'b'), ('b', 'c')):
    clear = tex[rgt].x0 - tex[lft].x1
    flag = 'OK' if clear > 0 else 'OVERLAP'
    print(f'  ({lft}) -> ({rgt}) clearance {clear:6.1f} pt   {flag}')
assert all(tex[r].x0 > tex[l].x1 for l, r in (('a', 'b'), ('b', 'c'))), 'panel titles overlap'

OUT_PNG = OUTPUT_DIR / 'fig2_linear_transfer.png'
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_PNG, dpi=300, bbox_inches='tight')
print()
print('saved:', OUT_PNG)